# The Supercharged Expansion: Auditing Tesla’s Market Penetration Through Infrastructure and Production Scaling

![Alt text](Tesla-prices-2025-lineup.jpg)

The introduction of the Model S created a seismic shift in the EV industry- EVs with niche technological features were now accessible challenging other mainstream established brands, proving EVs can be better than their gas powered countrerpats. This analysis examines Tesla's market penetration in different regions, by way of different models, and the supercharger density. The entry of other models will also be examined, together with factors determining price elasticity, operational efficiency, and supply chain scaling. 

## Business Questions
1. Operational Efficiency
- What is the Delivery-to-Production Ratio across regions, and can we identify specific 'Operational Bottlenecks' where production scaling outpaces logistics capacity?
- To what extent does End-of-Quarter Cyclicality impact delivery efficiency, and has Tesla successfully 'smoothed' this curve between 2015 and 2025?
2. Market penetration
- What is the Price Elasticity of Demand per region, and how does the entry of 'Mass-Market' models (3/Y) shift the correlation between Avg_Price_USD and Market_Share? 
- Does Supercharger Density act as a leading indicator for market penetration, and what is the 'Saturation Point' where additional stations no longer yield proportional delivery growth?
3. Sustainability
- What is the Carbon-Offset Efficiency (CO2 saved per kWh produced), and how has the regional energy mix transition influenced the 'Net-Positive' impact of Tesla's fleet over time?

## Key Performance Indicators


| Strategic Pillar | Primary KPI | Data Columns Involved |
| :--- | :--- | :--- |
| Operations | Delivery Ratio | Production_Units, Estimated_Deliveries |
| Penetration | Infrastructure Lead | Charging_Stations, YearMonth, Region |
| Elasticity | Price-to-Volume | Avg_Price_USD, Estimated_Deliveries, Model |


### Data Understanding

In [1]:
# Importing libraries
import polars as pl
import plotly as px

In [2]:
# importing the data from csv 
data = pl.read_csv('tesla_data.csv')

In [3]:
# Explore the general overview of the data, first 5 columns
data.head(5)

Year,Month,Region,Model,Estimated_Deliveries,Production_Units,Avg_Price_USD,Battery_Capacity_kWh,Range_km,CO2_Saved_tons,Source_Type,Charging_Stations
i64,i64,str,str,i64,i64,f64,i64,i64,f64,str,i64
2023,5,"""Europe""","""Model S""",17646,17922,92874.27,120,704,1863.42,"""Interpolated (Month)""",12207
2015,2,"""Asia""","""Model X""",3797,4164,62205.65,75,438,249.46,"""Official (Quarter)""",7640
2019,1,"""North America""","""Model X""",8411,9189,117887.32,82,480,605.59,"""Interpolated (Month)""",14071
2021,2,"""North America""","""Model 3""",6555,7311,89294.91,120,712,700.07,"""Official (Quarter)""",9333
2016,12,"""Middle East""","""Model Y""",12374,13537,114846.78,120,661,1226.88,"""Estimated (Region)""",8722


In [4]:
data.glimpse()

Rows: 2640
Columns: 12
$ Year                 <i64> 2023, 2015, 2019, 2021, 2016, 2020, 2015, 2020, 2022, 2021
$ Month                <i64> 5, 2, 1, 2, 12, 4, 11, 6, 4, 3
$ Region               <str> 'Europe', 'Asia', 'North America', 'North America', 'Middle East', 'Asia', 'Asia', 'Europe', 'Europe', 'Middle East'
$ Model                <str> 'Model S', 'Model X', 'Model X', 'Model 3', 'Model Y', 'Model X', 'Model 3', 'Cybertruck', 'Model S', 'Model Y'
$ Estimated_Deliveries <i64> 17646, 3797, 8411, 6555, 12374, 4656, 7717, 8410, 15145, 7790
$ Production_Units     <i64> 17922, 4164, 9189, 7311, 13537, 5043, 7976, 9192, 15760, 8208
$ Avg_Price_USD        <f64> 92874.27, 62205.65, 117887.32, 89294.91, 114846.78, 86930.57, 87588.21, 73815.61, 69993.86, 50591.6
$ Battery_Capacity_kWh <i64> 120, 75, 82, 120, 120, 82, 82, 100, 100, 82
$ Range_km             <i64> 704, 438, 480, 712, 661, 477, 475, 592, 563, 485
$ CO2_Saved_tons       <f64> 1863.42, 249.46, 605.59, 700.07, 1226.88, 333.14, 5

In [5]:
data.describe()

statistic,Year,Month,Region,Model,Estimated_Deliveries,Production_Units,Avg_Price_USD,Battery_Capacity_kWh,Range_km,CO2_Saved_tons,Source_Type,Charging_Stations
str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,str,f64
"""count""",2640.0,2640.0,"""2640""","""2640""",2640.0,2640.0,2640.0,2640.0,2640.0,2640.0,"""2640""",2640.0
"""null_count""",0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,"""0""",0.0
"""mean""",2020.0,6.5,null,null,9922.199621,10655.847348,84907.34033,87.05947,500.257576,744.076989,null,8932.133712
"""std""",3.162877,3.452707,null,null,3935.950093,4260.600858,20123.258036,20.836265,120.868549,353.221224,null,3469.565883
"""min""",2015.0,1.0,"""Asia""","""Cybertruck""",48.0,50.0,50003.7,60.0,330.0,3.07,"""Estimated (Region)""",3002.0
"""25%""",2017.0,4.0,null,null,7292.0,7829.0,67727.18,75.0,418.0,499.73,null,5899.0
"""50%""",2020.0,7.0,null,null,9860.0,10550.0,85073.32,82.0,470.0,699.65,null,8902.0
"""75%""",2023.0,9.0,null,null,12506.0,13469.0,102371.33,100.0,586.0,943.29,null,11937.0
"""max""",2025.0,12.0,"""North America""","""Model Y""",25704.0,28939.0,119965.36,120.0,719.0,2548.55,"""Official (Quarter)""",14996.0


In [6]:
# Investigating the columns in the data
data.columns

['Year',
 'Month',
 'Region',
 'Model',
 'Estimated_Deliveries',
 'Production_Units',
 'Avg_Price_USD',
 'Battery_Capacity_kWh',
 'Range_km',
 'CO2_Saved_tons',
 'Source_Type',
 'Charging_Stations']

In [7]:
data_lazy = pl.scan_csv("tesla_data.csv")

In [8]:
null_report = data_lazy.null_count().collect()

In [9]:
null_report

Year,Month,Region,Model,Estimated_Deliveries,Production_Units,Avg_Price_USD,Battery_Capacity_kWh,Range_km,CO2_Saved_tons,Source_Type,Charging_Stations
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0


In [10]:
# Ensure we have the counts (this works whether 'data' was eager or lazy)
# If eager, just use null_report = data.null_count()
null_report = data.null_count()

# Step-by-step logic to flag issues
print("--- TESLA DATA INTEGRITY AUDIT ---")
has_issues = False

for col in null_report.columns:
    # In Polars, we access the value in the first row [0]
    count = null_report[col][0]
    
    if count > 0:
        print(f"⚠️ AUDIT ALERT: There are {count} nulls on '{col}' column.")
        has_issues = True

if not has_issues:
    print("✅ DATA CLEAN: No nulls detected in any functional columns.")

--- TESLA DATA INTEGRITY AUDIT ---
✅ DATA CLEAN: No nulls detected in any functional columns.
